In [1]:
# import openai
import boto3
import requests
import pandas as pd
import json
from datetime import datetime, timedelta

In [2]:
# import os
# print(os.getcwd())

# # Print model names that bedrock supports.
# bedrock = boto3.client("bedrock", region_name="us-east-1")
# response = bedrock.list_foundation_models()
# print(response)

In [2]:
# Load data from CSV
def load_data(filepath):
    df = pd.read_csv(filepath, parse_dates=['Date'])
    df.sort_values(by='Date', inplace=True)
    return df

In [3]:
def format_prompt(data, n):
    prompt = f"""
     Human: As an AI central bank advisor, review the past {n} months of U.S. economic data and predict the Fed Funds Target Rate the FOMC is likely to set on {target_date}.
   Please respond in valid JSON with the following fields:
    {{
      "predicted_rate": "e.g. 5.25",
      "confidence": "e.g. 85%",
      "explanation": "Brief rationale, max 100 words"
    }}

    
 #    Human: Given the following {n} months of historical economic indicators (U3 unemployment rate, U6 unemployment rate,
 # 5 year inflation expectations, changes in nonfarm payroll, growth rate of money supply using M2 data, inflation rate of core CPI and PCE, 
 # growth rate of housing price and commercial real estate price, growth rate of hourly wage rate and lastly GDP growth),
 #    predict the Fed's decision. There have been growing concerns about uncertainties surrounding tariffs.
    
 #    Data:
 #    {data.to_string(index=False)}
    
    # Provide the following output in JSON format:
    # {{
    #     "decision": ["hawkish, neutral, dovish"],
    #     "confidence": ["percentage"],
    #     "magnitude": ["increase/decrease magnitude"],
    #     "explanation": ["max nn words"]
    # }}
    
    Assistant:
    """
    return prompt.strip()  # Ensure no extra spaces


In [5]:
# # Function to query OpenAI (ChatGPT)
# def query_chatgpt(prompt, api_key):
#     response = openai.ChatCompletion.create(
#         model="gpt-4-turbo",
#         messages=[{"role": "system", "content": "You are an AI analyzing Fed decisions."},
#                   {"role": "user", "content": prompt}],
#         api_key=api_key
#     )
#     return response["choices"][0]["message"]["content"]


In [4]:
# Function to query Anthropic Claude via AWS Bedrock
def query_claude_aws(prompt):
    bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
    
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",  # Required for Messages API
        "messages": [{"role": "user", "content": prompt}],  # Use messages instead of prompt
        "max_tokens": 500
    }
    
    # Invoke the model (use `invoke_model` instead of `invoke_model_with_response_stream`)
    response = bedrock.invoke_model(
        modelId="anthropic.claude-3-5-sonnet-20240620-v1:0",
        body=json.dumps(request_body).encode("utf-8")  # Ensure JSON is sent as bytes
    )

    # Read and decode the response
    return json.loads(response["body"].read().decode("utf-8"))

In [7]:

# Function to query DeepSeek via AWS Bedrock
# def query_deepseek_aws(prompt):
#     bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
    
#     response = bedrock.invoke_model(
#         modelId="deepseek.r1-v1:0",  # Correct DeepSeek model ID
#         body=json.dumps({"prompt": prompt, "max_tokens": 500})
#     )
    
#     return json.loads(response["body"].read().decode("utf-8"))

# # Example usage
# response = query_deepseek_aws("What is the capital of France?")
# print(response)


In [8]:
# # === Query Meta Llama 3 on AWS Bedrock ===
# def query_llama_aws(prompt, model_id="meta.llama3-8b-instruct-v1:0", max_tokens=500):
#     """
#     Queries the Meta Llama 3 model via AWS Bedrock.

#     Parameters:
#         prompt (str): The text input for the model.
#         model_id (str): The Bedrock model ID for Llama 3.
#         max_tokens (int): The maximum number of tokens to generate.

#     Returns:
#         dict: The model's JSON response.
#     """
#     # Initialize AWS Bedrock Runtime Client
#     bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

#     # Construct request body
#     payload = {
#         "prompt": prompt,
#         "max_gen_len": max_tokens  # Llama uses `max_gen_len`
#     }

#     # Invoke model
#     response = bedrock.invoke_model(
#         modelId=model_id,
#         body=json.dumps(payload)
#     )

#     # Read and parse response
#     response_body = json.loads(response["body"].read().decode("utf-8"))

#     return response_body  # Extract relevant text if necessary


In [5]:
# Main function to run monthly predictions
def run_predictions(filepath, n, api_keys):
    df = load_data(filepath)
    latest_date = df["Date"].max()
    start_date = latest_date - pd.DateOffset(months=n)
    filtered_data = df[df["Date"] >= start_date][['U3',
 'U6',
 'inflation.expectation.5year',
 'change.nonfarm.payroll',
 'M2_growth',
 'CPI.core_growth',
 'PCE.core_growth',
 'house.price_growth',
 'commercial.real.estate_growth',
 'wage.hourly_growth',
 'GDP_growth']]
    
    prompt = format_prompt(filtered_data, n)
    
    results = {
        # "chatgpt": query_chatgpt(prompt, api_keys["chatgpt"]),
        "claude": query_claude_aws(prompt),
        # "deepseek": query_deepseek_aws(prompt),
        # "llama3": query_llama_aws(prompt) #Llama cannot help, and provide DL code instead to set up a prediction model.
        # "grok": query_grok(prompt, api_keys["grok"]),
    }
    
    return results


In [6]:
# Example usage
if __name__ == "__main__":
    api_keys = {
        "chatgpt": "your_openai_api_key",
        "grok": "your_grok_api_key",
    }
    
    filepath = "Data_cleaned.csv"  # Replace with your actual file
    n = 6  # Number of months to look back
    predictions = run_predictions(filepath, n, api_keys)
    print(predictions)


NameError: name 'target_date' is not defined

[NbConvertApp] WARNING | pattern 'my_notebook.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
   